# Faz 3 — Colab ortamı ve Weymouth sayısal doğrulaması

**Amaç:** GitHub deposundan kodu alıp Colab ortamında bağımlılıkları kurmak, `pip install -e .` ve temel testleri çalıştırmak, ardından gerçek `fluids.compressible.Weymouth` çağrısıyla sayısal smoke testlerini yürütmek.

**Önemli sınırlar**
- Bu notebook model eğitimi veya nihai deney değildir.
- Paket konfigürasyonundaki bir sayı bilimsel kararın kapandığını göstermez.
- Başarılı API/geri çözüm smoke testleri, seçilen sabit `Zavg=0.8065` varsayımının fiziksel yeterliliğini veya sızıntı modelinin fiziksel yeterliliğini tek başına doğrulamaz. `OPEN-DECISION-012`, ilgili kanıtlar karar defterine işlenene kadar açık kalır.
- Secret/token hücreye yazılmaz. Özel depo için Colab Secrets bölümünde `GITHUB_TOKEN` adlı bir secret kullanılır; gerekli olduğunda yalnızca çalışma anında okunur.

Ön koşul: GitHub'da `natural-gas-anomaly-detection` deposu oluşturulmuş ve bu proje dosyaları `main` dalına push edilmiş olmalıdır.

## 1. Depo ayarları

Aşağıdaki `GITHUB_OWNER` değerini GitHub kullanıcı/organizasyon adınla değiştir. `REPO_NAME` ve `BRANCH` depo yapına göre değiştirilebilir. Özel depo kullanıyorsan Colab'da **Secrets** bölümünden `GITHUB_TOKEN` ekle; tokena `Contents: Read` yetkisi yeterlidir. Secret değerini notebook hücresine veya Git'e yazma.

In [ ]:
from pathlib import Path
import os
import subprocess
import tempfile

GITHUB_OWNER = "YOUR_GITHUB_USERNAME_OR_ORG"
REPO_NAME = "natural-gas-anomaly-detection"
BRANCH = "main"
PROJECT_ROOT = Path("/content") / REPO_NAME

if GITHUB_OWNER == "YOUR_GITHUB_USERNAME_OR_ORG":
    raise ValueError("GITHUB_OWNER değerini kendi GitHub kullanıcı/organizasyon adınla değiştir.")

if not (PROJECT_ROOT / "pyproject.toml").exists():
    repo_url = f"https://github.com/{GITHUB_OWNER}/{REPO_NAME}.git"
    env = os.environ.copy()
    env["GIT_TERMINAL_PROMPT"] = "0"
    askpass_dir = None
    try:
        # Optional private-repo authentication from Colab Secrets, not notebook literals.
        try:
            from google.colab import userdata
            gh_token = userdata.get("GITHUB_TOKEN")
        except Exception:
            gh_token = None

        if gh_token:
            askpass_dir = tempfile.TemporaryDirectory(prefix="git-askpass-")
            askpass_path = Path(askpass_dir.name) / "askpass.sh"
            askpass_path.write_text(
                '#!/bin/sh\ncase "$1" in\n'
                '  *Username*) printf "%s\n" "x-access-token" ;;\n'
                '  *Password*) printf "%s\n" "$GITHUB_TOKEN" ;;\n'
                '  *) exit 1 ;;\nesac\n',
                encoding="utf-8",
            )
            askpass_path.chmod(0o700)
            env["GIT_ASKPASS"] = str(askpass_path)
            env["GITHUB_TOKEN"] = gh_token
        subprocess.run(["git", "clone", "--branch", BRANCH, repo_url, str(PROJECT_ROOT)], check=True, env=env)
    finally:
        # Remove temporary askpass script and drop the in-memory variable reference.
        if askpass_dir is not None:
            askpass_dir.cleanup()
        # Remove the credential from the environment dictionary and notebook namespace.
        if "env" in globals():
            env.pop("GITHUB_TOKEN", None)
            env.pop("GIT_ASKPASS", None)
            del env
        if "gh_token" in globals():
            del gh_token

assert (PROJECT_ROOT / "pyproject.toml").exists(), f"Project root not found: {PROJECT_ROOT}"
print("Project root:", PROJECT_ROOT)
print("Current commit:")
commit_cmd = subprocess.run(
    ["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"],
    check=True, text=True, capture_output=True,
)
print(commit_cmd.stdout.strip())
if commit_cmd.stderr:
    print("STDERR:", commit_cmd.stderr)

## 2. Google Drive kalıcı çıktı klasörü

Colab sanal makinesindeki `/content` kalıcı değildir. Doğrulama raporlarını ve ortam manifestini Drive'a yazacağız. Proje kodu GitHub'ın kaynak kontrolünde kalacak; büyük veri/checkpoint dosyaları Git'e eklenmeyecek.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/natural-gas-anomaly-detection")
(DRIVE_ROOT / "reports" / "metrics").mkdir(parents=True, exist_ok=True)
print("Persistent report directory:", DRIVE_ROOT / "reports" / "metrics")

## 3. Bağımlılık kurulumu ve editable package

`fluids` için doğrulama amacıyla sürüm `1.3.1` sabitlenmiştir. Diğer bağımlılıklar bu ilk ortam smoke testinde başlangıç listesi olarak kurulur; nihai deneylerden önce Colab'da gerçekten çalışan sürümlerin manifesti kaydedilecektir.

**Colab uyarısı:** `pip install` sonrasında çalışma zamanı yeniden başlarsa notebook'u baştan çalıştır; önce depo ayarı/klonlama, ardından bağımlılık kurulum hücrelerinin tamamlandığını doğrula. Sürüm sabitlemesi ancak manifestte gerçekten çalışan sürümler görüldükten sonra yapılacaktır.


In [ ]:
%cd /content/natural-gas-anomaly-detection
%pip install -r requirements.txt
%pip install -e .

In [ ]:
import fluids
from fluids.compressible import Weymouth
import inspect
print("fluids version:", getattr(fluids, "__version__", "unknown"))
print("Weymouth signature:", inspect.signature(Weymouth))
print("Weymouth module:", Weymouth.__module__)

## 4. Temel testler

Bu testler depo yapısı ve Weymouth gerçek kütüphane çağrısı dâhil temel kontrol içindir. `test_weymouth_validation.py` Colab'da **skip olmadan çalışmalıdır**. Atlanan veya başarısız test, Faz 3'ün geçtiği anlamına gelmez.

In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    cwd=PROJECT_ROOT, text=True, capture_output=True,
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)
print("pytest exit code:", result.returncode)
if result.returncode != 0:
    raise RuntimeError("pytest başarısız oldu. Hataları düzeltmeden sonraki adıma geçme.")

## 5. Weymouth sayısal doğrulama

Kontroller:
1. `fluids` dokümantasyonundaki örnek sonucu yeniden üret.
2. Proje nominal noktasıyla önce `Q0` hesapla, sonra aynı akışı kullanarak `P2` değerini geri çöz.
3. Talep akışını ±%5 değiştirince aşağı akım basıncının sonlu/pozitif kalmasını ve akış artarken basıncın düşmesini doğrula.
4. Orta noktada sızıntı için `Qin = Qout + Qleak` kütle dengesini ve iki segmentin basınç sıralamasını kontrol et. v1 için sabit `Zavg=0.8065` seçilmiştir; smoke test tek başına OPEN-DECISION-012 sayısal kapanışını sağlamaz.

In [ ]:
from pathlib import Path
import json

OUTPUT_JSON = DRIVE_ROOT / "reports" / "metrics" / "weymouth_validation.json"
run = subprocess.run(
    [sys.executable, "scripts/run_weymouth_validation.py", "--output", str(OUTPUT_JSON)],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
)
print(run.stdout)
if run.stderr:
    print("STDERR:", run.stderr)
if run.returncode != 0:
    raise RuntimeError("Weymouth validation başarısız. OPEN-DECISION-012 kapatılmamalı.")

report = json.loads(OUTPUT_JSON.read_text(encoding="utf-8"))
assert report["all_passed"] is True
print("Report persisted to:", OUTPUT_JSON)

## 6. Ortam manifesti

Python/platform ve temel paket sürümlerini kaydet. Bu dosyayı nihai deneylerde de saklayacağız.

In [ ]:
manifest_cmd = subprocess.run(
    [sys.executable, "scripts/capture_environment.py"],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
    check=True,
)
print(manifest_cmd.stdout)
local_manifest = PROJECT_ROOT / "reports" / "metrics" / "phase3_environment_manifest.json"
persistent_manifest = DRIVE_ROOT / "reports" / "metrics" / "phase3_environment_manifest.json"
persistent_manifest.write_text(local_manifest.read_text(encoding="utf-8"), encoding="utf-8")
print("Persistent manifest:", persistent_manifest)

## 7. Faz 3 durumunu kaydet

Bu notebook bittiğinde şunları karar defterine ekle:
- GitHub repo URL'si, ilk commit SHA'sı ve ana dal adı.
- Colab çalışma tarihi/ortamı, Python ve `fluids` sürümü.
- `pip install -r requirements.txt`, `pip install -e .` ve `pytest -q` sonuçları.
- `weymouth_validation.json` içindeki her kontrolün çıktısı.

**Bilimsel kapanış kuralı:** Testler geçerse yalnızca kütüphane API'si, nominal geri çözüm ve temel sayısal davranış doğrulanmış olur. `OPEN-DECISION-012` ancak seçilen sabit `Zavg=0.8065` varsayımının sınırlılıkları ve sade sızıntı modelinin yeterliliği ayrıca gözden geçirilip kanıt/karar kaydı/dosya güncellemeleri tamamlanınca kapanabilir. Bu notebook tek başına kararı otomatik kapatmaz.